# 1. Imports

In [ ]:
!pip install implicit lightgbm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=933264 sha256=1a2aa5aad30262b1c593611fcc052a25195d9f5335a9a8d0c0924e34646a14b1
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [ ]:
!pip install rectools[lightfm,torch] lightning-fabric

In [ ]:
import os
import glob
import json

import pandas as pd
import numpy as np

from lightfm import LightFM

from rectools import Columns
from rectools.dataset import Dataset
from rectools.models import PopularModel, LightFMWrapperModel, SASRecModel
from rectools.metrics import NDCG, HitRate
from rectools.visuals.visual_app import ItemToItemVisualApp

from lightning_fabric import seed_everything

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Enable deterministic behaviour with CUDA >= 10.2
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
RANDOM_STATE = 42
seed_everything(RANDOM_STATE, workers=True)

DATA_DIR = "/content/drive/MyDrive/VK/"

INFO:lightning_fabric.utilities.seed:Seed set to 42


# 2. Load data

In [ ]:
train_paths = glob.glob(DATA_DIR + "train/*.json")
print("Train files:", len(train_paths))

train_data = pd.concat([
    pd.read_json(path, lines=True)[["user", "track", "timestamp", "time"]]
    for path in train_paths
])

train_data.head(3)

Train files: 2


,user,track,timestamp,time
0,2906,4455,2026-04-21 07:33:36.835,0.88
1,2906,7218,2026-04-21 07:33:36.847,0.92
2,136,9901,2026-04-21 07:33:36.879,1.00


In [ ]:
test_paths = glob.glob(DATA_DIR + "2026-03-07-8x-10k-full-random/*/data.json")
print("Test files:", len(test_paths))

test_data = pd.concat([
    pd.read_json(path, lines=True)[["user", "track", "timestamp", "time"]]
    for path in test_paths
])

test_data.head(3)

Test files: 8


,user,track,timestamp,time
0,8474,943,2026-03-07 07:19:28.488,1.00
1,667,4336,2026-03-07 07:19:28.502,0.09
2,3434,3076,2026-03-07 07:19:28.506,0.00


In [ ]:
train_df = train_data[train_data["time"] > 0.7].copy()

test_df = test_data[
    (test_data["time"] > 0.7)
    & (test_data["user"].isin(train_df["user"]))
    & (test_data["track"].isin(train_df["track"]))
].copy()

In [ ]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (195887, 4)
Test shape: (169550, 4)


,user,track,timestamp,time
0,2906,4455,2026-04-21 07:33:36.835,0.88
1,2906,7218,2026-04-21 07:33:36.847,0.92
2,136,9901,2026-04-21 07:33:36.879,1.00
8,9570,503,2026-04-21 07:33:36.924,1.00
12,9570,7171,2026-04-21 07:33:36.956,0.85


In [ ]:
train_df["user"] = train_df["user"].astype(int)
train_df["track"] = train_df["track"].astype(int)

test_df["user"] = test_df["user"].astype(int)
test_df["track"] = test_df["track"].astype(int)

# 3. Preprocessing

# 4. Train/Val split

# 5. ALS candidate generation

In [ ]:
import numpy as np

# уникальные пользователи и треки
unique_users = train_df["user"].unique()
unique_items = train_df["track"].unique()

# mapping → индекс
user2id = {u: i for i, u in enumerate(unique_users)}
item2id = {t: i for i, t in enumerate(unique_items)}

# обратные mapping (очень важно)
id2user = {i: u for u, i in user2id.items()}
id2item = {i: t for t, i in item2id.items()}

# применяем mapping
train_df["user_id"] = train_df["user"].map(user2id)
train_df["item_id"] = train_df["track"].map(item2id)

train_df.head()

,user,track,timestamp,time,user_id,item_id
0,2906,4455,2026-04-21 07:33:36.835,0.88,0,0
1,2906,7218,2026-04-21 07:33:36.847,0.92,0,1
2,136,9901,2026-04-21 07:33:36.879,1.00,1,2
8,9570,503,2026-04-21 07:33:36.924,1.00,2,3
12,9570,7171,2026-04-21 07:33:36.956,0.85,2,4


In [ ]:
from scipy.sparse import coo_matrix

alpha = 40

train_df["confidence"] = 1 + alpha * train_df["time"]

user_item_matrix = coo_matrix(
    (
        train_df["confidence"],
        (train_df["user_id"], train_df["item_id"])
    )
).tocsr()


In [ ]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=20,
    random_state=42
)

# ВАЖНО: transpose!
model.fit(user_item_matrix)

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
def get_als_candidates(user, N=80):
    if user not in user2id:
        return []

    user_id = user2id[user]

    item_ids, scores = model.recommend(
        userid=user_id,
        user_items=user_item_matrix[user_id],  # 🔥 ВАЖНО
        N=N,
        filter_already_liked_items=True
    )

    return [id2item[i] for i in item_ids]

In [ ]:
user_example = train_df["user"].iloc[0]

print("User:", user_example)
print("Candidates:", get_als_candidates(user_example)[:10])

User: 2906
Candidates: [14317, 3832, 3960, 4272, 956, 1754, 8164, 8166, 3935, 3946]


#Popular

In [ ]:
popular = (
    train_df["track"]
    .value_counts()
    .head(20)
    .index
    .tolist()
)

In [ ]:
def get_candidates(user):
    als = get_als_candidates(user, 80)

    combined = als.copy()
    for track in popular:
        if track not in combined:
            combined.append(track)

    return combined

In [ ]:
candidates = get_candidates(user_example)

print(len(candidates))   # ≈ 100
print(candidates[:10])

99
[14317, 3832, 3960, 4272, 956, 1754, 8164, 8166, 3935, 3946]


#TEST

In [ ]:
gt = (
    test_df.groupby("user")["track"]
    .apply(set)
    .to_dict()
)

In [ ]:
def hit_rate_at_k(get_candidates_func, k=10):
    hits = 0
    total = 0

    for user in gt.keys():
        recs = get_candidates_func(user)[:k]
        true_items = gt[user]

        if len(true_items.intersection(recs)) > 0:
            hits += 1

        total += 1

    return hits / total

In [ ]:
hr_als = hit_rate_at_k(lambda u: get_als_candidates(u, 80))
print("ALS HR@10:", hr_als)

ALS HR@10: 0.47735017037482463


In [ ]:
hr_pop = hit_rate_at_k(lambda u: popular, 10)
print("Popular HR@10:", hr_pop)

Popular HR@10: 0.05431950290639407


In [ ]:
hr_combined = hit_rate_at_k(get_candidates, 10)
print("Combined HR@10:", hr_combined)

Combined HR@10: 0.47735017037482463


#RANKING DATASET

In [ ]:
import numpy as np

embeddings = np.load(DATA_DIR + "embeddings.npy")

print(embeddings.shape)


(16198, 4096)


In [ ]:
embeddings[:5]

array([[ 0.02731  , -0.001688 ,  0.004837 , ...,  0.01076  , -0.01053  ,
         0.00818  ],
       [ 0.01846  ,  0.0001084,  0.02882  , ...,  0.02686  ,  0.00977  ,
         0.009476 ],
       [ 0.0075   , -0.01347  ,  0.003899 , ...,  0.01317  ,  0.00496  ,
        -0.006435 ],
       [-0.01009  , -0.012115 ,  0.03214  , ...,  0.001163 ,  0.01588  ,
         0.02066  ],
       [ 0.01996  ,  0.01377  ,  0.005398 , ...,  0.002726 , -0.00854  ,
        -0.00183  ]], dtype=float16)

In [ ]:
gt_dict = (
    test_df.groupby("user")
    .apply(lambda x: dict(zip(x["track"], x["time"])))
    .to_dict()
)

rows = []

for user in test_df["user"].unique():
    candidates = get_candidates(user)
    user_dict = gt_dict.get(user, {})

    for track in candidates:
        target = user_dict.get(track, 0)

        rows.append({
            "user": user,
            "track": track,
            "target": target
        })

rank_df = pd.DataFrame(rows)

/tmp/ipykernel_1148/1253983674.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x["track"], x["time"])))


In [ ]:
rank_df.head()
rank_df.shape

(986000, 3)

In [ ]:
user_features = train_df.groupby("user")["time"].agg([
    "mean", "count"
]).reset_index()

user_features.columns = [
    "user",
    "user_mean_time",
    "user_total_tracks"
]

In [ ]:
track_features = train_df.groupby("track")["time"].agg([
    "mean", "count"
]).reset_index()

track_features.columns = [
    "track",
    "track_mean_time",
    "track_popularity"
]

In [ ]:
def get_als_rank_dict(user):
    als = get_als_candidates(user, 80)
    return {track: rank for rank, track in enumerate(als)}

In [ ]:
als_rank_dict_all = {}

for user in rank_df["user"].unique():
    als = get_als_candidates(user, 80)
    als_rank_dict_all[user] = {
        track: rank for rank, track in enumerate(als)
    }

rank_df["als_rank"] = [
    als_rank_dict_all[user].get(track, 100)
    for user, track in zip(rank_df["user"], rank_df["track"])
]

In [ ]:
rank_df = rank_df.merge(user_features, on="user", how="left")
rank_df = rank_df.merge(track_features, on="track", how="left")

In [ ]:
rank_df["target_rank"] = (rank_df["target"] * 4).astype(int)

In [ ]:
import lightgbm as lgb

features = [
    "user_mean_time",
    "user_total_tracks",
    "track_mean_time",
    "track_popularity",
    "als_rank"
]

group = rank_df.groupby("user").size().values

train_data = lgb.Dataset(
    rank_df[features],
    label=rank_df["target_rank"],
    group=group
)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "learning_rate": 0.05,
    "num_leaves": 31
}

model_lgb = lgb.train(params, train_data)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016227 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 915
[LightGBM] [Info] Number of data points in the train set: 986000, number of used features: 5


In [ ]:
rank_df["score"] = model_lgb.predict(rank_df[features])

In [ ]:
recommendations = {}

for user, group_df in rank_df.groupby("user"):
    top_tracks = (
        group_df
        .sort_values("score", ascending=False)["track"]
        .head(100)
        .tolist()
    )
    recommendations[str(user)] = top_tracks

In [ ]:
import json

with open("/content/drive/MyDrive/VK/hstu_recommendations.json", "w") as f:
    json.dump(recommendations, f)

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/VK/hstu_recommendations.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
len(recommendations)
len(list(recommendations.values())[0])

100

In [ ]:
gt = (
    test_df.groupby("user")["track"]
    .apply(set)
    .to_dict()
)

In [ ]:
def hit_rate_at_k_dict(recommendations, k=10):
    hits = 0
    total = 0

    for user in gt.keys():
        if str(user) not in recommendations:
            continue

        recs = recommendations[str(user)][:k]
        true_items = gt[user]

        if len(true_items.intersection(recs)) > 0:
            hits += 1

        total += 1

    return hits / total

In [ ]:
als_recommendations = {}

for user in test_df["user"].unique():
    als_recommendations[str(user)] = get_candidates(user)[:100]

In [ ]:
hr_als = hit_rate_at_k_dict(als_recommendations, 10)
hr_lgbm = hit_rate_at_k_dict(recommendations, 10)

print("ALS HR@10:", hr_als)
print("LGBM HR@10:", hr_lgbm)

ALS HR@10: 0.47735017037482463
LGBM HR@10: 0.5116255762677892


# 6. Feature engineering

# 7. Dataset for ranking

# 8. Train LightGBM

# 9. Validation

# 10. Inference

# 11. Save JSON